# Module 14: Python for Machine Learning

**Lesson: Supervised Learning, Unsupervised Learning, and Hyperparameter Tuning**

This notebook covers the complete ML workflow using scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, fetch_california_housing, load_digits
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, mean_squared_error, r2_score
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded successfully')

## 1. Scikit-Learn API and Train/Test Split

Every sklearn model follows the same pattern: `model.fit(X, y)` then `model.predict(X_test)`.

In [ ]:
# Load Iris dataset
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')
print(f'Features: {X_train.shape[1]}')
print('\nTarget distribution in train:')
print(y_train.value_counts().sort_index())

In [ ]:
# Example: Logistic Regression
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print('=== Logistic Regression on Iris ===')
print(f'Accuracy: {accuracy:.4f}')
print(f'Coefficients shape: {model.coef_.shape}')
print(f'Intercept: {model.intercept_}')

## 2. Cross-Validation

Cross-validation gives a more reliable estimate of model performance by using multiple train/validation splits.

In [ ]:
# 5-fold cross-validation
scores = cross_val_score(LogisticRegression(max_iter=200, random_state=42), X, y, cv=5)

print('=== 5-Fold Cross-Validation Scores ===')
for i, score in enumerate(scores, 1):
    print(f'Fold {i}: {score:.4f}')
print(f'\nMean: {scores.mean():.4f}')
print(f'Std: {scores.std():.4f}')

## 3. Regression with California Housing

Linear Regression and comparison of regression metrics.

In [ ]:
# Load California Housing
housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_h_train_scaled = scaler.fit_transform(X_h_train)
X_h_test_scaled = scaler.transform(X_h_test)

# Linear Regression
lr = LinearRegression()
lr.fit(X_h_train_scaled, y_h_train)
y_pred_lr = lr.predict(X_h_test_scaled)

print('=== Linear Regression on California Housing ===')
print(f'MSE: {mean_squared_error(y_h_test, y_pred_lr):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_h_test, y_pred_lr)):.4f}')
print(f'MAE: {np.mean(np.abs(y_h_test - y_pred_lr)):.4f}')
print(f'R2: {r2_score(y_h_test, y_pred_lr):.4f}')

# Feature coefficients
coef_df = pd.DataFrame({'feature': housing.feature_names, 'coefficient': lr.coef_})
print('\nTop coefficients:')
print(coef_df.sort_values('coefficient', key=abs, ascending=False))

## 4. Classification — Model Comparison

Compare Logistic Regression, Decision Tree, Random Forest, SVM, and KNN on Titanic.

In [ ]:
# Load and prepare Titanic
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].dropna()
titanic['sex'] = (titanic['sex'] == 'male').astype(int)
titanic = pd.get_dummies(titanic, columns=['embarked'], drop_first=True)

X_t = titanic.drop('survived', axis=1)
y_t = titanic['survived']

X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)

scaler = StandardScaler()
X_t_train_scaled = scaler.fit_transform(X_t_train)
X_t_test_scaled = scaler.transform(X_t_test)

print('Titanic data prepared.')
print(f'Train: {X_t_train.shape}, Test: {X_t_test.shape}')

In [ ]:
# Compare classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = []
for name, clf in classifiers.items():
    clf.fit(X_t_train_scaled, y_t_train)
    y_pred = clf.predict(X_t_test_scaled)
    y_prob = clf.predict_proba(X_t_test_scaled)[:, 1] if hasattr(clf, 'predict_proba') else None
    
    result = {
        'Model': name,
        'Accuracy': accuracy_score(y_t_test, y_pred),
        'Precision': precision_score(y_t_test, y_pred),
        'Recall': recall_score(y_t_test, y_pred),
        'F1': f1_score(y_t_test, y_pred),
    }
    if y_prob is not None:
        result['ROC-AUC'] = roc_auc_score(y_t_test, y_prob)
    results.append(result)

results_df = pd.DataFrame(results)
print('=== Model Comparison on Titanic ===')
print(results_df.round(4))

## 5. Unsupervised Learning — K-Means, DBSCAN, PCA

In [ ]:
# K-Means clustering on Iris (without labels)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)

print('=== K-Means on Iris ===')
print(f'Inertia: {kmeans.inertia_:.2f}')
print('Cluster centers:')
print(pd.DataFrame(kmeans.cluster_centers_, columns=iris.feature_names).round(2))

# Compare with true labels
contingency = pd.crosstab(y, kmeans_labels, rownames=['True'], colnames=['Cluster'])
print('\nClustering contingency:')
print(contingency)

In [ ]:
# PCA for dimensionality reduction on Digits dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_digits)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits, cmap='tab10', alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.title('PCA: Digits Dataset (64d -> 2d)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
print(f'Explained variance ratio: {pca.explained_variance_ratio_.sum():.1%}')

In [ ]:
# DBSCAN for density-based clustering on synthetic data
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.1, random_state=42)

dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)

kmeans_labels_2 = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=dbscan_labels, cmap='Set1', s=50)
axes[0].set_title('DBSCAN Clustering')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_labels_2, cmap='Set1', s=50)
axes[1].set_title('K-Means Clustering')
plt.show()
print('DBSCAN handles non-spherical clusters better than K-Means.')

## 6. Hyperparameter Tuning with GridSearchCV

In [ ]:
# GridSearchCV on Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0)
grid_search.fit(X_t_train_scaled, y_t_train)

print('=== GridSearchCV Results ===')
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best cross-val F1: {grid_search.best_score_:.4f}')

# Evaluate on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_t_test_scaled)
print(f'Test F1: {f1_score(y_t_test, y_pred_best):.4f}')
print(f'Test Accuracy: {accuracy_score(y_t_test, y_pred_best):.4f}')

## Summary

- **scikit-learn API**: `fit()`, `predict()`, `transform()` are universal
- **Cross-validation**: More reliable than single train/test split
- **Supervised learning**: Choose model based on data size, linearity, and interpretability needs
- **Unsupervised learning**: K-Means (spherical clusters), DBSCAN (arbitrary shapes), PCA (dimensionality reduction)
- **Metrics**: Match metric to problem — F1 for imbalanced, ROC-AUC for ranking, MSE for regression
- **Hyperparameter tuning**: GridSearchCV (exhaustive) vs RandomizedSearchCV (efficient for large spaces)
- **Key ML workflow**: Clean → Split → Scale → Train → Tune → Evaluate — always keep test data untouched until the end